## 08 - TIMER Deconvolution (split out from notebook 04's plan)

### Aim
* Run the real, original TIMER algorithm (`hanfeisun/TIMER`, the exact GitHub repo originally specified) on the TCGA-PAAD cohort, to cross-validate against XDec (notebook 06) and CIBERSORTx (notebook 07). Decision: run the actual archived repo directly rather than the `immunedeconv` wrapper - `r-immunedeconv` turned out to be uninstallable via `pixi` (its dependency `r-xcell` has no bioconda build for R 4.5.3, only up to R 4.4), so this notebook uses TIMER's own `regression.r`/`utils.r` instead.

### Input
* `data/processed/TCGA-PAAD_TPM_raw.tsv` (notebook 02) - TPM, not log-transformed, `gene_symbol` row index. TIMER's `ParseInputExpression()` expects exactly this (tab-separated, gene IDs as row names - optionally `Symbol|EntrezID`, but plain symbols work fine since the `|`-split is a no-op without a `|`).
* `tools/TIMER/` (cloned `hanfeisun/TIMER`) + `tools/TIMER/data/precalculated/` and `data/immune_datasets/` (extracted from the repo's own `prec.zip`, fetched from the Dropbox link in its README - confirmed still live despite being a ~2016 link).

### Output
* `data/deconvolution/timer_real_full/results/score_matrix.txt` - 6 immune cell types (B_cell, T_cell.CD4, T_cell.CD8, Neutrophil, Macrophage, DC) x 182 samples, real ComBat-batch-corrected fractions against TIMER's own PAAD-specific reference (`genes_paad.RData`).
* `data/deconvolution/timer_real_full/results/output.pdf` - the per-sample Q-Q diagnostic plots TIMER generates as part of its own batch-correction QC.

### Steps
1. Clone `hanfeisun/TIMER`, extract its `prec.zip` (precalculated PAAD marker genes + curated immune reference - confirmed present, no need for the separate `data.zip`).
2. **Dependency conflict found and fixed**: `utils.r` calls `library(sqldf)` at the top, but `r-sqldf`'s dependency `r-chron` has no bioconda build for R 4.5.3 (same root cause as the `immunedeconv`/`r-xcell` block). Traced the code path: `sqldf` is only used inside `ConvertImmuneProbeToRefgene()`, which is skipped whenever the precalculated `immune.expression.curated.RData` already exists (it does). Patched the local clone's `utils.r` to not load `sqldf` - zero risk, since the function that needs it is never reached on this path.
3. Verified on a 5-sample test slice first (real ComBat correction, real fractions, no errors) before committing to the full cohort.
4. Ran the full 182-sample cohort via `Rscript regression.r <bulk TPM> paad`.

### Result
* **Done.** All four steps executed successfully, no unresolved errors. Real output at `data/deconvolution/timer_real_full/results/score_matrix.txt`.

### Conclusion
* TIMER is the one of the three tools with a fully completed run. XDec (notebook 06) has its gene set but still needs the k-sweep + Stage 1/2; CIBERSORTx (notebook 07) is blocked on a token.

In [1]:
if (basename(getwd()) == "rnaseq_download") setwd(file.path("..", ".."))
getwd()

[1] "/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress"

### Step 1-2: clone, extract `prec.zip`, patch `utils.r` (idempotent - skips if already done)

In [2]:
timer_repo <- "tools/TIMER"
if (!file.exists(file.path(timer_repo, "regression.r"))) {
  system2("git", c("clone", "--depth", "1", "https://github.com/hanfeisun/TIMER.git", timer_repo))
}

prec_marker <- file.path(timer_repo, "data", "precalculated", "genes_paad.RData")
if (!file.exists(prec_marker)) {
  prec_zip <- file.path(timer_repo, "prec.zip")
  system2("curl", c("-sL", "-o", prec_zip,
                     "https://www.dropbox.com/s/qiwcfdnx1xp90j8/prec.zip?dl=1"))
  unzip(prec_zip, exdir = timer_repo)
}
stopifnot(file.exists(prec_marker))

utils_path <- file.path(timer_repo, "utils.r")
utils_lines <- readLines(utils_path)
if (any(grepl("^library\\(sqldf\\)", utils_lines))) {
  utils_lines[grepl("^library\\(sqldf\\)", utils_lines)] <- paste(
    "# sqldf intentionally not loaded: only used inside ConvertImmuneProbeToRefgene(),",
    "# which is skipped whenever data/precalculated/immune.expression.curated.RData",
    "# exists (it does, from prec.zip) - sqldf's dependency chain (r-chron) has no",
    "# bioconda build compatible with this environment's R 4.5.3, and isn't needed",
    "# on this code path.",
    sep = "\n"
  )
  writeLines(utils_lines, utils_path)
}
message("TIMER repo ready at ", timer_repo)

TIMER repo ready at tools/TIMER



### Steps 3-4: run TIMER's `regression.r` on the full TCGA-PAAD cohort

In [3]:
timer_outdir <- normalizePath("data/deconvolution/timer_real_full", mustWork = FALSE)
dir.create(timer_outdir, recursive = TRUE, showWarnings = FALSE)
score_path <- file.path(timer_outdir, "results", "score_matrix.txt")

if (!file.exists(score_path)) {
  rscript_bin <- path.expand("~/.pixi/bin/Rscript")
  tpm_path <- normalizePath("data/processed/TCGA-PAAD_TPM_raw.tsv")

  old_wd <- getwd()
  setwd(timer_repo)
  on.exit(setwd(old_wd), add = TRUE)

  timer_log <- system2(rscript_bin, c(
    "regression.r", tpm_path, "paad",
    paste0("--outdir=", timer_outdir)
  ), stdout = TRUE, stderr = TRUE)

  setwd(old_wd)
  cat(tail(timer_log, 20), sep = "\n")
}

stopifnot(file.exists(score_path))
timer_res <- read.delim(score_path, check.names = FALSE)
timer_res[, 1:5]

,TCGA.IB.7888.01A.11R.2156.07,TCGA.FB.A78T.01A.12R.A32O.07,TCGA.2J.AABI.01A.12R.A41B.07,TCGA.HZ.7918.01A.11R.2156.07
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
B_cell,0.1922291,0.067393577,0.12380927,0.15457401
T_cell.CD4,0.1330308,0.151659492,0.07074316,0.10145014
T_cell.CD8,0.3011092,0.100023784,0.24100455,0.33697190
Neutrophil,0.1448789,0.103292956,0.15333303,0.11233317
Macrophage,0.1244364,0.004380269,0.00000000,0.09362661
DC,0.6615943,0.458142176,0.51622079,0.77000245
